In [2]:
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import numpy as np
import pickle
from neuralhydrology.evaluation.metrics import calculate_all_metrics
from neuralhydrology.evaluation.metrics import calculate_metrics
from neuralhydrology.evaluation.metrics import missed_peaks
import neuralhydrology

In [4]:
# ------------------- Paths -------------------
# RUN_DIR = Path("./runs") #Uruguay
RUN_DIR = Path("../extending_caravan/runs") #USA

# Ensemble metrics

In [5]:
# URUGUAY

# run_pattern = "precip_prcp_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*" 

# matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
# print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

In [38]:
# USA

# run_pattern = "154_camels_30_epochs_seq_270_hidden_256_dropout_04_fb_05_seed111_1103_063301" # CAMELS
# run_pattern = "caravan_154_precip_seq_270_30_epochs_hidden_256_dropout_04_fb_05_seed111_0103_211409" # CARAVAN
# run_pattern = "precip_chirps_precipitation_0503_093821" # CHIRPS
# run_pattern = "precip_mswep_precipitation_0503_110038" # MSWEP
# run_pattern = "precip_total_precipitation_sum_chirps_precipitation_0303_072447" # CARAVAN + CHIRPS
# run_pattern = "precip_total_precipitation_sum_mswep_precipitation_0303_084843" # CARAVAN + MSWEP
# run_pattern = "precip_chirps_precipitation_mswep_precipitation_0303_101255" # CHIRPS + MSWEP
run_pattern = "precip_total_precipitation_sum_chirps_precipitation_mswep_precipitation_0303_113714" # CARAVAN + CHIRPS + MSWEP

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 1 runs: ['precip_total_precipitation_sum_chirps_precipitation_mswep_precipitation_0303_113714']


In [39]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

# ensemble_data

In [40]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    all_metrics[basin_id] = calculate_metrics(
        obs=xr_ds['streamflow_obs'], #['QObs_mm_d_obs'],
        sim=xr_ds['streamflow_sim'], #['QObs_mm_d_sim'],
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'

df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.672626,0.557557,0.746697,0.554242,0.643598,0.891339,0.755315,-0.245934,-40.211422,-1.111523e+01,3.464795e+01,0.150000,0.452830,41.694199
camels_01487000,0.806473,0.265689,0.515450,0.879461,0.950618,0.900719,1.047268,0.048281,-3.277431,-8.804256e+00,-1.642958e+01,0.166667,0.428571,20.673563
camels_01491000,0.607004,1.727968,1.314522,0.545982,0.587501,0.810352,0.996704,-0.001877,-41.308567,-1.380581e+01,7.188703e+01,0.388889,0.450980,63.208286
camels_01644000,0.763215,0.909475,0.953664,0.836296,0.921087,0.875659,0.928509,-0.038734,-0.714747,7.151977e-01,-8.877997e+00,0.500000,0.283019,36.728905
camels_01664000,0.462673,2.425890,1.557527,0.723832,1.018143,0.737499,0.916140,-0.049693,-3.909214,-2.091469e+00,6.471882e+01,0.142857,0.474576,48.537342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_08200000,0.686209,1.856153,1.362407,0.699586,0.828890,0.829106,0.821772,-0.034748,-34.143456,-4.635884e+01,-3.024232e+01,0.250000,0.324324,57.907051
camels_08202700,0.591522,0.649612,0.805985,-3.619292,1.299593,0.888954,5.608228,0.173421,89.441246,1.026755e+08,-1.155736e+12,0.500000,0.285714,55.612045
camels_09484600,0.064038,0.003118,0.055837,-0.621979,0.577373,0.407327,2.449462,0.270118,-44.089718,6.646314e+07,-9.502188e+11,0.384615,0.534884,68.994583


In [41]:
save_name = run_pattern #.split("_seed_")[0]
df_metrics.to_csv(f"./all_ensemble_metrics/{save_name}.csv")